# Stage 5: Train GNN Models

Trains and compares four approaches on the reasoning graphs:

1. **Text Baseline** — logistic regression on averaged step embeddings, ignores graph structure
2. **Graph Feature Baseline** — logistic regression on graph statistics, ignores text
3. **GCN** — Graph Convolutional Network, treats all edges equally
4. **GAT** — Graph Attention Network, learns different attention weights per edge type

All GNN models use Focal Loss with proper per-class alpha to handle the 75/25 class imbalance. Each model is trained three times with different random seeds and results are averaged.

All models are evaluated at the **default 0.5 decision threshold (argmax)** so the comparison is fair across methods. We report Accuracy, weighted F1, AUC-ROC, and PR-AUC. AUC-ROC and PR-AUC are threshold-independent and are the more reliable metrics on this imbalanced dataset.

**Input:** `data/graphs.pt`
**Output:** `data/gcn_model.pt`, `data/gat_model.pt`, `data/results.json`


## Cell 1 — Install libraries

We need `scikit-learn` for evaluation metrics and `scipy` for paired t-tests.
After this cell run **Kernel → Restart**, then run from the next cell.


In [ ]:
import sys
\!{sys.executable} -m pip install -q scikit-learn scipy
print('Done — Kernel → Restart, then run from the next cell')


## Cell 2 — Imports and file paths

In [ ]:
import os
import json
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F
from torch_geometric.loader import DataLoader as GeoDataLoader
from torch_geometric.nn import GCNConv, GATConv, global_mean_pool
from sklearn.metrics import (
    accuracy_score, f1_score, roc_auc_score, average_precision_score
)

DATA_DIR     = os.path.join(os.getcwd(), 'data')
GRAPHS_FILE  = os.path.join(DATA_DIR, 'graphs.pt')
GCN_MODEL    = os.path.join(DATA_DIR, 'gcn_model.pt')
GAT_MODEL    = os.path.join(DATA_DIR, 'gat_model.pt')
RESULTS_FILE = os.path.join(DATA_DIR, 'results.json')

device = torch.device('cpu')

print('Imports OK')
print(f'Graphs:  {GRAPHS_FILE}')
print(f'Device:  {device}')
print('graphs.pt found' if os.path.exists(GRAPHS_FILE) else 'graphs.pt NOT found - run Stage 4 first')


## Cell 3 — Load graphs and split into train/val/test

Class imbalance is handled by Focal Loss with per-class alpha during training.


In [ ]:
all_graphs = torch.load(GRAPHS_FILE, weights_only=False)

train_graphs = [g for g in all_graphs if g.split == 'train']
val_graphs   = [g for g in all_graphs if g.split == 'val']
test_graphs  = [g for g in all_graphs if g.split == 'test']

print(f'Total graphs: {len(all_graphs)}')
print(f'Train:  {len(train_graphs)}')
print(f'Val:    {len(val_graphs)}')
print(f'Test:   {len(test_graphs)}')
print()

for name, gs in [('Train', train_graphs), ('Val', val_graphs), ('Test', test_graphs)]:
    n1 = sum(1 for g in gs if g.y.item() == 1)
    n0 = sum(1 for g in gs if g.y.item() == 0)
    print(f'{name}: correct={n1} ({n1/max(len(gs),1)*100:.0f}%)  wrong={n0} ({n0/max(len(gs),1)*100:.0f}%)')

print()
print(f'Node feature size: {all_graphs[0].x.shape[1]}')
print(f'Avg nodes/graph:   {sum(g.x.shape[0] for g in all_graphs)/len(all_graphs):.1f}')
print(f'Avg edges/graph:   {sum(g.edge_index.shape[1] for g in all_graphs)/len(all_graphs):.1f}')
print('\nData loaded.')


## Cell 4 — GCN and GAT model architectures

Both follow the same structure: two conv layers (768 → 128 → 64), global mean pool, MLP classifier.
GAT uses `edge_dim=1` so it can learn different attention per edge type.


In [ ]:
class GCNModel(nn.Module):
    def __init__(self, input_dim=768, hidden_dim=128, out_dim=64, num_classes=2, dropout=0.3):
        super().__init__()
        self.conv1 = GCNConv(input_dim, hidden_dim)
        self.conv2 = GCNConv(hidden_dim, out_dim)
        self.mlp   = nn.Sequential(
            nn.Linear(out_dim, 32), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(32, num_classes)
        )
        self.dropout = dropout

    def forward(self, x, edge_index, batch, edge_attr=None):
        # GCNConv ignores edge_attr by design
        x = F.relu(self.conv1(x, edge_index))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.conv2(x, edge_index))
        x = global_mean_pool(x, batch)
        return self.mlp(x)


class GATModel(nn.Module):
    def __init__(self, input_dim=768, hidden_dim=128, out_dim=64, num_classes=2, heads=4, dropout=0.3):
        super().__init__()
        self.conv1 = GATConv(input_dim, hidden_dim // heads, heads=heads,
                             edge_dim=1, dropout=dropout)
        self.conv2 = GATConv(hidden_dim, out_dim, heads=1, concat=False,
                             edge_dim=1, dropout=dropout)
        self.mlp   = nn.Sequential(
            nn.Linear(out_dim, 32), nn.ReLU(),
            nn.Dropout(dropout), nn.Linear(32, num_classes)
        )
        self.dropout = dropout

    def forward(self, x, edge_index, batch, edge_attr=None):
        x = F.relu(self.conv1(x, edge_index, edge_attr=edge_attr))
        x = F.dropout(x, p=self.dropout, training=self.training)
        x = F.relu(self.conv2(x, edge_index, edge_attr=edge_attr))
        x = global_mean_pool(x, batch)
        return self.mlp(x)


sample = train_graphs[0]
batch_vec = torch.zeros(sample.x.shape[0], dtype=torch.long)
ea = getattr(sample, 'edge_attr', None)

g = GCNModel(); a = GATModel()
print(f'GCN out: {g(sample.x, sample.edge_index, batch_vec, ea).shape}  params={sum(p.numel() for p in g.parameters()):,}')
print(f'GAT out: {a(sample.x, sample.edge_index, batch_vec, ea).shape}  params={sum(p.numel() for p in a.parameters()):,}')
if ea is not None:
    print(f'Edge types in sample: {sorted(set(ea.squeeze().tolist())) if ea.numel() else []}')


## Cell 5 — Training and evaluation functions

**Focal Loss** with **per-class alpha**: minority class (wrong, label=0) gets weight `alpha`; majority class gets `1 - alpha`. This is the conventional form. A single global scalar alpha (as in some online tutorials) does not actually re-balance classes.

**Evaluation** uses the **default 0.5 threshold via argmax** for ALL methods (GNN and baselines). We report Accuracy, weighted F1, AUC-ROC and PR-AUC. The PR-AUC is computed for the WRONG class (label 0), which is the minority class we actually care about detecting.


In [ ]:
class FocalLoss(nn.Module):
    """Per-class alpha focal loss for binary classification.

    alpha = weight on minority class (label=0, 'wrong'). 1-alpha goes to majority.
    gamma = focusing parameter; higher gamma down-weights easy examples more.
    """
    def __init__(self, alpha=0.75, gamma=2.0):
        super().__init__()
        self.alpha = alpha
        self.gamma = gamma

    def forward(self, logits, targets):
        ce = F.cross_entropy(logits, targets, reduction='none')
        pt = torch.exp(-ce)
        alpha_t = torch.where(targets == 0,
                              torch.full_like(ce, self.alpha),
                              torch.full_like(ce, 1.0 - self.alpha))
        return (alpha_t * (1.0 - pt) ** self.gamma * ce).mean()


def train_epoch(model, graphs, optimizer, criterion):
    model.train()
    total = 0
    loader = GeoDataLoader(graphs, batch_size=16, shuffle=True)
    for batch in loader:
        batch = batch.to(device)
        optimizer.zero_grad()
        out  = model(batch.x, batch.edge_index, batch.batch,
                     getattr(batch, 'edge_attr', None))
        loss = criterion(out, batch.y)
        loss.backward()
        optimizer.step()
        total += loss.item()
    return total / max(len(loader), 1)


def evaluate(model, graphs, criterion):
    model.eval()
    preds_all, probs_all, labels_all = [], [], []
    total = 0
    loader = GeoDataLoader(graphs, batch_size=16, shuffle=False)
    with torch.no_grad():
        for batch in loader:
            batch = batch.to(device)
            out = model(batch.x, batch.edge_index, batch.batch,
                        getattr(batch, 'edge_attr', None))
            total += criterion(out, batch.y).item()
            probs_correct = F.softmax(out, dim=1)[:, 1]
            preds_all.extend(out.argmax(dim=1).cpu().numpy())
            probs_all.extend(probs_correct.cpu().numpy())
            labels_all.extend(batch.y.cpu().numpy())
    acc = accuracy_score(labels_all, preds_all)
    f1  = f1_score(labels_all, preds_all, average='weighted', zero_division=0)
    try:
        auc = roc_auc_score(labels_all, probs_all)
    except Exception:
        auc = float('nan')
    try:
        pr_auc = average_precision_score(1 - np.array(labels_all),
                                         1 - np.array(probs_all))
    except Exception:
        pr_auc = float('nan')
    return total / max(len(loader), 1), acc, f1, auc, pr_auc


def train_model(model_class, train_g, val_g, test_g, seed=42, epochs=200, lr=5e-4, patience=30):
    torch.manual_seed(seed); np.random.seed(seed)
    model     = model_class().to(device)
    optimizer = torch.optim.Adam(model.parameters(), lr=lr, weight_decay=1e-4)
    criterion = FocalLoss(alpha=0.75, gamma=2.0)

    best_val_loss, best_state, patience_count = float('inf'), None, 0

    for epoch in range(1, epochs + 1):
        tr_loss = train_epoch(model, train_g, optimizer, criterion)
        v_loss, v_acc, v_f1, v_auc, v_pr = evaluate(model, val_g, criterion)
        if v_loss < best_val_loss:
            best_val_loss = v_loss
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
            patience_count = 0
        else:
            patience_count += 1
        if epoch % 10 == 0:
            print(f'  ep{epoch:3d} train={tr_loss:.4f} val_loss={v_loss:.4f} acc={v_acc:.3f} f1={v_f1:.3f} auc={v_auc:.3f} pr={v_pr:.3f}')
        if patience_count >= patience:
            print(f'  Early stop at epoch {epoch}')
            break

    model.load_state_dict(best_state)
    _, t_acc, t_f1, t_auc, t_pr = evaluate(model, test_g, criterion)
    return model, t_acc, t_f1, t_auc, t_pr


print('FocalLoss (per-class alpha) and training/eval functions defined.')
print('Eval uses argmax (0.5 threshold). Reports Acc, F1, AUC-ROC, PR-AUC.')


## Cell 6 — Train text baseline

Logistic regression on averaged step embeddings. Ignores graph structure. Establishes the floor.
Evaluated at default 0.5 threshold, same as GNN models.


In [ ]:
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import StandardScaler

def graphs_to_flat_features(graphs):
    X = np.array([g.x.numpy().mean(axis=0) for g in graphs])
    y = np.array([g.y.item() for g in graphs])
    return X, y

print('Training text baseline...\n')
baseline_accs, baseline_f1s, baseline_aucs, baseline_prs = [], [], [], []

for seed in [42, 123, 456]:
    X_tr_, y_tr_ = graphs_to_flat_features(train_graphs)
    X_v,   y_v   = graphs_to_flat_features(val_graphs)
    X_t,   y_t   = graphs_to_flat_features(test_graphs)
    X_tr = np.vstack([X_tr_, X_v])
    y_tr = np.concatenate([y_tr_, y_v])

    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_tr)
    X_t_s  = scaler.transform(X_t)

    clf = LogisticRegression(max_iter=1000, random_state=seed, class_weight='balanced')
    clf.fit(X_tr_s, y_tr)
    preds = clf.predict(X_t_s)
    probs = clf.predict_proba(X_t_s)[:, 1]

    acc = accuracy_score(y_t, preds)
    f1  = f1_score(y_t, preds, average='weighted', zero_division=0)
    auc = roc_auc_score(y_t, probs) if len(set(y_t)) > 1 else float('nan')
    pr_auc = average_precision_score(1 - y_t, 1 - probs) if len(set(y_t)) > 1 else float('nan')

    baseline_accs.append(acc); baseline_f1s.append(f1)
    baseline_aucs.append(auc); baseline_prs.append(pr_auc)
    print(f'  seed {seed}: acc={acc:.3f}  f1={f1:.3f}  auc={auc:.3f}  pr={pr_auc:.3f}')

print('\nText Baseline (mean ± std over 3 seeds):')
print(f'  Accuracy: {np.mean(baseline_accs):.3f} ± {np.std(baseline_accs):.3f}')
print(f'  F1:       {np.mean(baseline_f1s):.3f} ± {np.std(baseline_f1s):.3f}')
print(f'  AUC-ROC:  {np.mean(baseline_aucs):.3f} ± {np.std(baseline_aucs):.3f}')
print(f'  PR-AUC:   {np.mean(baseline_prs):.3f} ± {np.std(baseline_prs):.3f}')


## Cell 7 — Train graph-statistics baseline

Logistic regression on graph-level structural features only. No embeddings.
This is a hard baseline — if GNN doesn't beat it, neural graph learning isn't adding value over simple structural statistics.


In [ ]:
def graph_to_feature_vector(g):
    n_nodes = g.x.shape[0]
    n_edges = g.edge_index.shape[1]
    avg_deg = n_edges / max(n_nodes, 1)
    if g.edge_attr is not None and g.edge_attr.shape[0] > 0:
        attrs = g.edge_attr.squeeze().tolist()
        if isinstance(attrs, float): attrs = [attrs]
        T = max(len(attrs), 1)
        p_seq = sum(1 for a in attrs if int(a)==0)/T
        p_sem = sum(1 for a in attrs if int(a)==1)/T
        p_val = sum(1 for a in attrs if int(a)==2)/T
        p_men = sum(1 for a in attrs if int(a)==3)/T
    else:
        p_seq=p_sem=p_val=p_men=0.0
    density = n_edges / max(n_nodes*(n_nodes-1), 1)
    return np.array([n_nodes, n_edges, avg_deg, p_seq, p_sem, p_val, p_men, density], dtype=np.float32)

def train_graph_feature_baseline(train_g, val_g, test_g, seed=42):
    np.random.seed(seed)
    X_tr_ = np.array([graph_to_feature_vector(g) for g in train_g])
    X_v   = np.array([graph_to_feature_vector(g) for g in val_g])
    X_t   = np.array([graph_to_feature_vector(g) for g in test_g])
    y_tr_ = np.array([g.y.item() for g in train_g])
    y_v   = np.array([g.y.item() for g in val_g])
    y_t   = np.array([g.y.item() for g in test_g])
    X_tr  = np.vstack([X_tr_, X_v]); y_tr = np.concatenate([y_tr_, y_v])

    scaler = StandardScaler()
    X_tr_s = scaler.fit_transform(X_tr); X_t_s = scaler.transform(X_t)
    clf = LogisticRegression(max_iter=1000, random_state=seed, class_weight='balanced')
    clf.fit(X_tr_s, y_tr)
    preds = clf.predict(X_t_s)
    probs = clf.predict_proba(X_t_s)[:, 1]
    acc = accuracy_score(y_t, preds)
    f1  = f1_score(y_t, preds, average='weighted', zero_division=0)
    auc = roc_auc_score(y_t, probs) if len(set(y_t)) > 1 else float('nan')
    pr_auc = average_precision_score(1 - y_t, 1 - probs) if len(set(y_t)) > 1 else float('nan')
    return acc, f1, auc, pr_auc


print('Training graph feature baseline...\n')
gf_accs, gf_f1s, gf_aucs, gf_prs = [], [], [], []
for seed in [42, 123, 456]:
    acc, f1, auc, pr = train_graph_feature_baseline(train_graphs, val_graphs, test_graphs, seed=seed)
    gf_accs.append(acc); gf_f1s.append(f1); gf_aucs.append(auc); gf_prs.append(pr)
    print(f'  seed {seed}: acc={acc:.3f}  f1={f1:.3f}  auc={auc:.3f}  pr={pr:.3f}')

print('\nGraph Feature Baseline (mean ± std over 3 seeds):')
print(f'  Accuracy: {np.mean(gf_accs):.3f} ± {np.std(gf_accs):.3f}')
print(f'  F1:       {np.mean(gf_f1s):.3f} ± {np.std(gf_f1s):.3f}')
print(f'  AUC-ROC:  {np.mean(gf_aucs):.3f} ± {np.std(gf_aucs):.3f}')
print(f'  PR-AUC:   {np.mean(gf_prs):.3f} ± {np.std(gf_prs):.3f}')


## Cell 8 — Train GCN (3 seeds)

Best checkpoint per seed selected by validation loss. Best-of-3 saved by **validation AUC-ROC** (threshold-independent).

Expected time: ~5–10 min on CPU.


In [ ]:
import time
print('Training GCN (3 seeds)...\n' + '='*60)
gcn_accs, gcn_f1s, gcn_aucs, gcn_prs = [], [], [], []
best_gcn, best_gcn_auc = None, -1

t0 = time.time()
for seed in [42, 123, 456]:
    print(f'\nSeed {seed}:')
    model, acc, f1, auc, pr = train_model(GCNModel, train_graphs, val_graphs, test_graphs,
                                          seed=seed, epochs=200, lr=5e-4, patience=30)
    gcn_accs.append(acc); gcn_f1s.append(f1); gcn_aucs.append(auc); gcn_prs.append(pr)
    print(f'  Test: acc={acc:.3f} f1={f1:.3f} auc={auc:.3f} pr={pr:.3f}')
    if auc > best_gcn_auc:
        best_gcn_auc, best_gcn = auc, model

print('\n' + '='*60)
print('GCN Results (mean ± std over 3 seeds):')
print(f'  Accuracy: {np.mean(gcn_accs):.3f} ± {np.std(gcn_accs):.3f}')
print(f'  F1:       {np.mean(gcn_f1s):.3f} ± {np.std(gcn_f1s):.3f}')
print(f'  AUC-ROC:  {np.mean(gcn_aucs):.3f} ± {np.std(gcn_aucs):.3f}')
print(f'  PR-AUC:   {np.mean(gcn_prs):.3f} ± {np.std(gcn_prs):.3f}')
print(f'  Time:     {(time.time()-t0)/60:.1f} min')

torch.save(best_gcn.state_dict(), GCN_MODEL)
print(f'\nBest GCN (by AUC-ROC) saved: {GCN_MODEL}')


## Cell 9 — Train GAT (3 seeds)

Same training protocol as GCN. GAT has access to `edge_attr` (the edge type) so it can in principle learn different attention per edge type.

Expected time: ~8–12 min on CPU.


In [ ]:
print('Training GAT (3 seeds)...\n' + '='*60)
gat_accs, gat_f1s, gat_aucs, gat_prs = [], [], [], []
best_gat, best_gat_auc = None, -1

t0 = time.time()
for seed in [42, 123, 456]:
    print(f'\nSeed {seed}:')
    model, acc, f1, auc, pr = train_model(GATModel, train_graphs, val_graphs, test_graphs,
                                          seed=seed, epochs=200, lr=5e-4, patience=30)
    gat_accs.append(acc); gat_f1s.append(f1); gat_aucs.append(auc); gat_prs.append(pr)
    print(f'  Test: acc={acc:.3f} f1={f1:.3f} auc={auc:.3f} pr={pr:.3f}')
    if auc > best_gat_auc:
        best_gat_auc, best_gat = auc, model

print('\n' + '='*60)
print('GAT Results (mean ± std over 3 seeds):')
print(f'  Accuracy: {np.mean(gat_accs):.3f} ± {np.std(gat_accs):.3f}')
print(f'  F1:       {np.mean(gat_f1s):.3f} ± {np.std(gat_f1s):.3f}')
print(f'  AUC-ROC:  {np.mean(gat_aucs):.3f} ± {np.std(gat_aucs):.3f}')
print(f'  PR-AUC:   {np.mean(gat_prs):.3f} ± {np.std(gat_prs):.3f}')
print(f'  Time:     {(time.time()-t0)/60:.1f} min')

torch.save(best_gat.state_dict(), GAT_MODEL)
print(f'\nBest GAT (by AUC-ROC) saved: {GAT_MODEL}')


## Cell 10 — Comparison table, paired t-tests, save results

Reports paired t-tests across the 3 seeds. **Caveat:** N=3 gives very low statistical power. Treat p-values as directional only, not as evidence of significance.


In [ ]:
from scipy import stats as sp_stats

print('=' * 78)
print('  FINAL RESULTS (default 0.5 threshold; mean over 3 seeds)')
print('=' * 78)
print(f"{'Model':<28} {'Accuracy':>10} {'F1':>10} {'AUC-ROC':>10} {'PR-AUC':>10}")
print('-' * 78)
print(f"{'Text Baseline':<28} {np.mean(baseline_accs):>10.3f} {np.mean(baseline_f1s):>10.3f} {np.mean(baseline_aucs):>10.3f} {np.mean(baseline_prs):>10.3f}")
print(f"{'Graph Feature Baseline':<28} {np.mean(gf_accs):>10.3f} {np.mean(gf_f1s):>10.3f} {np.mean(gf_aucs):>10.3f} {np.mean(gf_prs):>10.3f}")
print(f"{'GCN':<28} {np.mean(gcn_accs):>10.3f} {np.mean(gcn_f1s):>10.3f} {np.mean(gcn_aucs):>10.3f} {np.mean(gcn_prs):>10.3f}")
print(f"{'GAT':<28} {np.mean(gat_accs):>10.3f} {np.mean(gat_f1s):>10.3f} {np.mean(gat_aucs):>10.3f} {np.mean(gat_prs):>10.3f}")
print('=' * 78)
print()

def paired_p(a, b):
    try:
        _, p = sp_stats.ttest_rel(a, b)
        return p
    except Exception:
        return float('nan')

print('Paired t-tests across 3 seeds (low power; directional only):')
print(f'  GCN vs Text Baseline (AUC-ROC):           p = {paired_p(gcn_aucs, baseline_aucs):.3f}')
print(f'  GCN vs Graph Feature Baseline (AUC-ROC):  p = {paired_p(gcn_aucs, gf_aucs):.3f}')
print(f'  GAT vs Graph Feature Baseline (AUC-ROC):  p = {paired_p(gat_aucs, gf_aucs):.3f}')
print(f'  GAT vs GCN (AUC-ROC):                     p = {paired_p(gat_aucs, gcn_aucs):.3f}')

def pack(accs, f1s, aucs, prs):
    return {
        'accuracy': float(np.mean(accs)), 'accuracy_std': float(np.std(accs)),
        'f1':       float(np.mean(f1s)),  'f1_std':       float(np.std(f1s)),
        'auc_roc':  float(np.mean(aucs)), 'auc_roc_std':  float(np.std(aucs)),
        'pr_auc':   float(np.mean(prs)),  'pr_auc_std':   float(np.std(prs)),
        'per_seed': {'accuracy': list(map(float, accs)),
                     'f1':       list(map(float, f1s)),
                     'auc_roc':  list(map(float, aucs)),
                     'pr_auc':   list(map(float, prs))}
    }

results = {
    'evaluation_protocol': {
        'threshold': 'default (argmax / 0.5)',
        'seeds': [42, 123, 456],
        'metrics': ['accuracy', 'f1_weighted', 'auc_roc', 'pr_auc'],
        'note': 'PR-AUC and AUC-ROC computed for the WRONG class (label 0) - the minority class.'
    },
    'text_baseline':          pack(baseline_accs, baseline_f1s, baseline_aucs, baseline_prs),
    'graph_feature_baseline': pack(gf_accs, gf_f1s, gf_aucs, gf_prs),
    'gcn':                    pack(gcn_accs, gcn_f1s, gcn_aucs, gcn_prs),
    'gat':                    pack(gat_accs, gat_f1s, gat_aucs, gat_prs),
}

with open(RESULTS_FILE, 'w') as f:
    json.dump(results, f, indent=2)
print(f'\nResults saved: {RESULTS_FILE}')


## Cell 11 — Quality checks

In [ ]:
print('=' * 55)
print('  STAGE 5 QUALITY CHECKS')
print('=' * 55)

ok = True
print('\n[1] Output files saved:')
for path, name in [(GCN_MODEL,'gcn_model.pt'), (GAT_MODEL,'gat_model.pt'), (RESULTS_FILE,'results.json')]:
    e = os.path.exists(path)
    print(f'     {"OK" if e else "MISSING"} {name}')
    ok = ok and e

print('\n[2] All methods AUC-ROC >= 0.5 (better than random ranking):')
for name, aucs in [('Text', baseline_aucs), ('GraphFeat', gf_aucs), ('GCN', gcn_aucs), ('GAT', gat_aucs)]:
    m = float(np.mean(aucs))
    print(f'     {name:<10} AUC-ROC = {m:.3f}  {"OK" if m >= 0.5 else "BELOW RANDOM"}')

print('\n[3] Per-metric winner across the four methods:')
methods = {
    'Text':      (baseline_accs, baseline_f1s, baseline_aucs, baseline_prs),
    'GraphFeat': (gf_accs, gf_f1s, gf_aucs, gf_prs),
    'GCN':       (gcn_accs, gcn_f1s, gcn_aucs, gcn_prs),
    'GAT':       (gat_accs, gat_f1s, gat_aucs, gat_prs),
}
for i, metric in enumerate(['Accuracy','F1','AUC-ROC','PR-AUC']):
    winner = max(methods.items(), key=lambda kv: float(np.mean(kv[1][i])))
    print(f'     {metric:<8} winner = {winner[0]} ({np.mean(winner[1][i]):.3f})')

print('\n[4] Results JSON valid:')
with open(RESULTS_FILE) as f:
    res = json.load(f)
required = ['text_baseline','graph_feature_baseline','gcn','gat']
keys_ok = all(k in res for k in required)
print(f'     {"OK" if keys_ok else "FAIL"} - keys present: {list(res.keys())}')

print('\n' + '=' * 55)
print('Stage 5 complete.' if ok else 'Some checks failed - see above.')


## Stage 5 done

**Output files in `data/`:**

| File           | Contents                       | Used by |
|----------------|--------------------------------|---------|
| `gcn_model.pt` | Best GCN weights (by val AUC)  | Stage 7 |
| `gat_model.pt` | Best GAT weights (by val AUC)  | Stage 7 |
| `results.json` | All metrics per model per seed | Stage 6 |

**How to read these results:**

- **Lead with AUC-ROC and PR-AUC.** They are threshold-independent and the most reliable comparison on this 75/25 imbalanced dataset.
- Accuracy is reported but is sensitive to threshold and class prior. Don't lean on accuracy alone.
- With 3 seeds and ~30 test graphs, standard deviations are wide and paired t-tests have very low power. Treat between-method gaps cautiously.

**Next:** Stage 6 produces charts and confusion matrices. Stage 7 runs GNNExplainer.
